# Gap Analysis

## Objective

This notebook studies overnight gaps and their relationship with subsequent intraday returns.

Overnight price movements often reflect information released while markets are closed.

## Research Questions

- Do gaps tend to continue or reverse?
- Does gap size affect future returns?
- Can gaps improve intraday forecasting models?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("./NIFTY_50_minute.csv")

df['date'] = pd.to_datetime(
    df['date'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('date')
df = df.set_index('date')

df.head()

,open,high,low,close,volume
date,,,,,
2015-01-09 09:15:00,8285.45,8295.90,8285.45,8292.10,0
2015-01-09 09:16:00,8292.60,8293.60,8287.20,8288.15,0
2015-01-09 09:17:00,8287.40,8293.90,8287.40,8293.90,0
2015-01-09 09:18:00,8294.25,8300.65,8293.90,8300.65,0
2015-01-09 09:19:00,8300.60,8301.30,8298.75,8301.20,0


In [2]:
# ============================================================
# DATA CLEANING
# ============================================================

market_open = pd.Timestamp("09:15").time()
market_close = pd.Timestamp("15:29").time()

df = df[
    (df.index.time >= market_open) &
    (df.index.time <= market_close)
].copy()

df = df[
    ~df.index.duplicated(keep="first")
]

print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

Rows: 974705
Start: 2015-01-09 09:15:00
End: 2025-07-25 15:29:00


# Gap Construction

The overnight gap is calculated as the difference between the current day's opening price and the previous day's closing price.

Gap size is normalized to allow comparisons across different market levels.

In [3]:
research_gap = []

days = list(df.groupby(df.index.date))

for i in range(1, len(days)):

    prev_day, prev_df = days[i-1]
    curr_day, curr_df = days[i]

    prev_close_bar = prev_df.between_time(
        "15:15",
        "15:15"
    )

    open_bar = curr_df.between_time(
        "09:15",
        "09:15"
    )

    eleven_bar = curr_df.between_time(
        "11:00",
        "11:00"
    )

    close_bar = curr_df.between_time(
        "15:15",
        "15:15"
    )

    if (
        len(prev_close_bar)==0
        or len(open_bar)==0
        or len(eleven_bar)==0
        or len(close_bar)==0
    ):
        continue

    prev_close = prev_close_bar.iloc[0]['close']

    day_open = open_bar.iloc[0]['open']

    eleven_close = eleven_bar.iloc[0]['close']

    day_close = close_bar.iloc[0]['close']

    gap_return = (
        day_open / prev_close - 1
    )

    morning_return = (
        eleven_close / day_open - 1
    )

    afternoon_return = (
        day_close / eleven_close - 1
    )

    research_gap.append({
        'date': pd.Timestamp(curr_day),
        'gap_return': gap_return,
        'morning_return': morning_return,
        'afternoon_return': afternoon_return
    })

research_gap = pd.DataFrame(research_gap)

research_gap.head()

,date,gap_return,morning_return,afternoon_return
0,2015-01-12,0.000712,-0.000476,0.004006
1,2015-01-13,0.003071,-0.000821,-0.004329
2,2015-01-14,0.000488,-0.000536,-0.003926
3,2015-01-15,0.018742,-0.000771,0.010637
4,2015-01-20,0.003106,0.003079,0.010388


# Gap Bucket Analysis

Gap observations are grouped according to magnitude.

The objective is to determine whether larger gaps lead to different intraday outcomes than smaller gaps.

In [4]:
bins = [
    -np.inf,
    -0.01,
    -0.005,
    0,
    0.005,
    0.01,
    np.inf
]

labels = [
    "< -1%",
    "-1% to -0.5%",
    "-0.5% to 0%",
    "0% to 0.5%",
    "0.5% to 1%",
    "> 1%"
]

research_gap['gap_bucket'] = pd.cut(
    research_gap['gap_return'],
    bins=bins,
    labels=labels
)

gap_results = (
    research_gap
    .groupby('gap_bucket')
    ['afternoon_return']
    .agg(['mean','median','count'])
)

print(gap_results)

                  mean    median  count
gap_bucket                             
< -1%         0.000848  0.000652     78
-1% to -0.5% -0.000471 -0.000506    137
-0.5% to 0%  -0.000237 -0.000194    604
0% to 0.5%   -0.000278 -0.000114   1335
0.5% to 1%    0.000613  0.000949    341
> 1%         -0.000990  0.000824     96


In [5]:
strong_momentum = research_gap[
    research_gap['morning_return'].abs() > 0.0075
]

strong_momentum.groupby(
    'gap_bucket'
)['afternoon_return'].agg(
    ['mean','median','count']
)

,mean,median,count
gap_bucket,,,
< -1%,0.001157,-0.001588,25
-1% to -0.5%,-0.000665,0.000038,21
-0.5% to 0%,-0.000608,-0.000232,42
0% to 0.5%,0.000096,-0.000099,89
0.5% to 1%,0.002232,0.002148,32
> 1%,0.001028,0.003372,24


# Conclusions

## Research Question

Do overnight gaps predict intraday returns?

## Evidence

- Gap size influences intraday behavior.
- Extreme gaps show different characteristics from normal trading days.
- Predictability is inconsistent across regimes.

## Verdict

🟡 Partially Accepted

Gap information contains useful context but is insufficient as a standalone forecasting signal.

Gaps are better used as an additional feature alongside other signals.